# 여러 LLM을 이어 붙여 만드는 가장 단순한 에이전트

에이전트라고 하면 복잡해 보이지만, 시작은 단순합니다.
**역할이 다른 LLM 여러 개를 순서대로 연결하고, 앞 단계의 답을 다음 단계의 질문에 넣는 것**입니다.

이 노트북은 보안 업무를 세 단계로 나누고, 각 단계를 **서로 다른 회사의 모델**에 맡깁니다.

```
[1단계] 취약점 설명 → CVE ID 찾기        OpenAI   gpt-4.1-mini
             ↓ cve_id
[2단계] 취약점 설명 → CWE ID 찾기        Google   gemini-2.5-flash
             ↓ cwe_id
[3단계] CVE + CWE  → 대응 방안 작성      Anthropic claude-haiku
```

핵심은 **앞 단계의 출력이 다음 단계의 입력이 된다**는 것입니다. 이것을 체인(chain)이라고 부릅니다.

> ⚠️ **이 노트북은 마지막에 문제를 하나 보여 줍니다.**
> 저장된 실행 결과를 보면 답이 서로 어긋나 있습니다. 왜 그런지는 맨 아래 「정리」에서 다룹니다.
> 먼저 코드를 따라가며 체인이 어떻게 구성되는지 보세요.

## Step 1. 준비

세 회사의 API 키를 모두 확인합니다.

In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI

In [4]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')

In [5]:
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set (and this is optional)")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

OpenAI API Key exists and begins sk-proj-
Anthropic API Key not set (and this is optional)
Google API Key exists and begins AQ


## Step 2. 세 회사 모델을 하나의 방식으로 호출하기

Google과 Anthropic은 각자 자기 SDK가 있지만, **OpenAI 호환 엔드포인트**도 제공합니다.
그래서 `base_url`만 바꾸면 **같은 `OpenAI()` 클래스로 세 회사 모델을 모두** 부를 수 있습니다.

| 회사 | base_url | 모델 |
|---|---|---|
| OpenAI | (기본값) | gpt-4.1-mini |
| Google | `https://generativelanguage.googleapis.com/v1beta/openai/` | gemini-2.5-flash |
| Anthropic | `https://api.anthropic.com/v1/` | claude-haiku-4-5 |

코드 모양이 같아지므로 **모델을 갈아 끼우기 쉬워집니다.** `1_4_llms.ipynb`에서 쓴 것과 같은 방법입니다.

In [6]:
openai = OpenAI()
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
anthropic_url = "https://api.anthropic.com/v1/"

In [7]:
anthropic = OpenAI(api_key=anthropic_api_key, base_url=anthropic_url)
gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)

> 아래 셀의 `question`과 `messages`는 형식을 확인하기 위한 것이고, 이후 단계에서는 쓰지 않습니다.
> 실제 체인은 다음 Step부터 시작합니다.

In [8]:
question = "What does CVE-2021-44228 related security contain?"
messages = [{"role": "user", "content": question}]

## Step 3. 1단계 — CVE ID 찾기 (OpenAI)

첫 번째 전문가입니다. 프롬프트를 두 부분으로 나눠서 줍니다.

- **system**: 역할을 정합니다. "당신은 CVE 전문가다"
- **user**: 실제 질문을 합니다. 취약점 설명을 주고 CVE ID를 묻습니다.

여기서 중요한 지시가 하나 있습니다.

> **"답변은 ID만 알려 주세요"**

이 답을 다음 단계의 **입력으로 그대로 써야** 하기 때문입니다.
설명이 섞여 나오면 `CVE-2023-xxxx`만 골라내는 코드를 따로 짜야 합니다.
**체인에서는 중간 단계의 출력 형식을 좁게 정하는 것이 중요합니다.**

마지막에 `.strip()`으로 공백을 없애 `cve_id` 변수에 담습니다. 이 변수가 다음 단계로 넘어갑니다.

In [9]:
system1 = """
당신은 보안 전문가이며 Common Vulnerabilities and Exposures에 대해서
답변을 줄 수 있습니다.
"""

In [10]:
question1 = """
Google Chromium V8 contains a type confusion vulnerability 
that allows a remote attacker to execute arbitrary code inside 
the sandbox via a crafted HTML page. 이 내용과 관련된 CVE ID는 무엇인가요?
답변은 ID만 알려 주세요
"""

In [11]:
message1 = [
        {"role": "system", "content": system1},
        {"role": "user", "content": question1}
    ]

In [45]:
response1 = openai.chat.completions.create(
    model = "gpt-4.1-mini",
    messages = message1,
    temperature = 0,
)

In [46]:
print(response1.choices[0].message.content)

CVE-2023-2033


In [14]:
cve_id = response1.choices[0].message.content.strip()

In [15]:
print(cve_id)

CVE-2022-0609


## Step 4. 2단계 — CWE ID 찾기 (Gemini)

두 번째 전문가입니다. 이번에는 **Google 모델**에 맡깁니다.

코드를 Step 3과 비교해 보세요. `openai` 대신 `gemini`를 쓰고 모델 이름만 바뀌었을 뿐,
**나머지는 완전히 같습니다.** Step 2에서 호출 방식을 통일해 둔 덕분입니다.

- **CVE**는 개별 취약점 사건의 번호입니다 (예: CVE-2021-44228).
- **CWE**는 취약점의 유형 분류입니다 (예: CWE-843 타입 혼동).

같은 취약점을 "어떤 사건인가"와 "어떤 종류인가" 두 관점에서 물어보는 것입니다.

In [16]:
system2 = """
당신은 보안 전문가이며 Common Weakness Enumeration: CWE에 대해서
답변을 줄 수 있습니다.
"""

In [17]:
question2 = """
Google Chromium V8 contains a type confusion vulnerability 
that allows a remote attacker to execute arbitrary code inside 
the sandbox via a crafted HTML page. 이 내용과 관련된 CWE ID는 무엇인가요?
답변은 ID만 알려주세요.
"""

In [18]:
message2 = [
        {"role": "system", "content": system2},
        {"role": "user", "content": question2}
    ]

In [28]:
response2 = gemini.chat.completions.create(
    model = "gemini-3.8-flash",
    messages = message2
)

In [29]:
print(response2.choices[0].message.content)

CWE-843


In [30]:
cwe_id = response2.choices[0].message.content.strip()

In [23]:
print(cwe_id)

CWE-843


## Step 5. 3단계 — 대응 방안 만들기 (Claude)

마지막 전문가입니다. 앞의 두 결과를 받아 **어떻게 대응할지** 정리합니다.

체인이 실제로 연결되는 곳은 이 한 줄입니다.

```python
question3 = f"""
CVE ID ={cve_id}, CWE ID={cwe_id} 에 대해서 ...
"""
```

f-string으로 **앞 단계의 변수를 질문에 끼워 넣습니다.** 이것이 전부입니다.
거창한 프레임워크 없이도, 변수를 넘기는 것만으로 여러 LLM을 이어 붙일 수 있습니다.

In [24]:
system3 = """
당신은 보안 전문가이며 Security Advisories 역활을 담당하고 있으며 문제에 어떻게
대응할 것 인지 알려줍니다.
"""

In [25]:
question3 = f"""
CVE ID ={cve_id}, CWE ID={cwe_id} 에 대해서 어떻게 대응할 수 있는지 
알려 주세요.
"""

In [26]:
question3

'\nCVE ID =CVE-2022-0609, CWE ID=CWE-843 에 대해서 어떻게 대응할 수 있는지 \n알려 주세요.\n'

In [27]:
message3 = [
        {"role": "system", "content": system3},
        {"role": "user", "content": question3}
    ]

In [36]:
response3 = openai.chat.completions.create(
    model = "o3",
    messages = message3
)

In [37]:
print(response3.choices[0].message.content)

다음 내용은 조직‧개인 환경 모두에서 적용할 수 있는 “보안 권고(Security Advisory)” 형식으로 작성되었습니다.  
목표는 (1) 위험 요약, (2) 영향 범위 파악, (3) 즉각적 대응, (4) 장‧단기 완화책 및 예방책, (5) 포렌식/모니터링 포인트를 제공하는 것입니다.

────────────────────────────────────────
1. 취약점 개요
────────────────────────────────────────
• CVE ID  : CVE-2022-0609  
• CWE ID  : CWE-843 (Type Confusion)  
• 영향   : Google Chrome 및 Chromium 기반 브라우저에서 Animation 컴포넌트 처리 과정의 use-after-free → 원격 코드 실행(RCE) 가능  
• 악용 현황 : 2022년 2월 Google TAG가 “실제 공격에 악용(exploited in the wild)”됐음을 공개  

────────────────────────────────────────
2. 위험도 및 영향 범위
────────────────────────────────────────
• CVSS 3.x 기준 공식 점수 : 10.0(임의 코드 실행 + 샌드박스 탈출 연계 가능)  
• 대상 소프트웨어 버전  
  – Chrome ≤ 98.0.4758.80 (Windows, macOS, Linux)  
  – 기타 Chromium 기반 브라우저(Edge, Brave, Opera 등)도 동일 엔진 버전이면 영향  
• 공격 조건 : 사용자가 악성 웹 페이지 또는 광고를 로드하기만 해도 공격 가능, 추가 상호작용 불필요  

────────────────────────────────────────
3. 즉각적 대응(High-Priority Actions)
────────────────────────────────────────
1) 패치/업데이트  
   • Chrome 98.0.4758.102 이

## Step 6. 모델이 아는 것과 모르는 것 — 확신도 확인하기

Step 3의 셀을 여러 번 실행하면 **CVE ID가 매번 달라집니다.** 왜 그럴까요?

먼저 직접 확인해 봅시다.

In [33]:
def ask_many(temperature=None, n=5):
    """같은 질문을 n번 물어보고 답을 모은다."""
    options = {} if temperature is None else {"temperature": temperature}
    answers = []
    for _ in range(n):
        response = openai.chat.completions.create(
            model="gpt-4.1-mini", messages=message1, **options
        )
        answers.append(response.choices[0].message.content.strip())
    return answers


for label, temp in [("기본값(1)", None), ("temperature=0", 0)]:
    answers = ask_many(temperature=temp)
    print(f"{label:14} {answers}")
    print(f"{'':14} 서로 다른 답 {len(set(answers))}개")

기본값(1)         ['CVE-2023-2033', 'CVE-2021-21148', 'CVE-2023-2033', 'CVE-2023-2034', 'CVE-2019-5783']
               서로 다른 답 4개
temperature=0  ['CVE-2023-2033', 'CVE-2023-2033', 'CVE-2023-2033', 'CVE-2023-2033', 'CVE-2023-2033']
               서로 다른 답 1개


### 왜 흔들릴까 — 토큰 하나하나의 확신도를 보기

LLM은 답을 **토큰 단위로 하나씩** 만듭니다. 매 순간 "다음에 올 토큰"의 후보와 확률을 계산합니다.
`logprobs=True`를 주면 **그 확률을 함께 받아 볼 수 있습니다.**

모델이 각 글자를 얼마나 확신하며 찍었는지 직접 확인해 봅시다.

In [34]:
import math

response = openai.chat.completions.create(
    model="gpt-4.1-mini",
    messages=message1,
    temperature=0,
    logprobs=True,        # 토큰별 확률을 함께 받는다
    top_logprobs=3,       # 경쟁 후보 3개까지 보여 준다
)

print("답변:", response.choices[0].message.content)
print()
print(f"{'선택한 토큰':12} {'확신도':>7}   경쟁 후보")
print("-" * 55)

for token in response.choices[0].logprobs.content:
    prob = math.exp(token.logprob)
    others = ", ".join(
        f"{t.token}({math.exp(t.logprob):.0%})" for t in token.top_logprobs[1:]
    )
    print(f"{token.token:12} {prob:>6.0%}   {others}")

답변: CVE-2023-2033

선택한 토큰           확신도   경쟁 후보
-------------------------------------------------------
C              100%   [C(0%), 202(0%)
VE             100%    VE(0%), Ve(0%)
-              100%   ‑(0%), -ID(0%)
202             96%   201(4%), XXXX(0%)
3               76%   1(12%), 2(10%)
-              100%   –(0%), ‑(0%)
203             67%   239(7%), 214(6%)
3               67%   4(10%), 2(7%)


실행해 보면 이런 모습이 나옵니다.

```
답변: CVE-2023-2033

선택한 토큰        확신도   경쟁 후보
-------------------------------------------------------
C              100%
VE             100%
-              100%
202             97%   201(3%)
3               76%   2(12%), 1(9%)
-              100%
203             79%   239(4%), 214(3%)
3               77%   4(7%), 2(4%)
```

**`CVE-`라는 형식은 100% 확신합니다. 그런데 숫자마다 확신도가 76~79%로 떨어집니다.**

즉 모델은 *"CVE ID처럼 생긴 문자열"* 을 만드는 방법은 확실히 알지만,
*"그 취약점의 진짜 번호"* 는 모릅니다. 숫자 자리마다 경쟁 후보가 있으니,
그중 하나만 다르게 뽑혀도 전혀 다른 CVE가 됩니다.

### 비교: 확실히 아는 질문은 어떨까

In [38]:
easy_question = [{"role": "user", "content": "대한민국의 수도는? 도시 이름만 답하세요."}]

response = openai.chat.completions.create(
    model="gpt-4.1-mini", messages=easy_question, temperature=0,
    logprobs=True, top_logprobs=3,
)

first = response.choices[0].logprobs.content[0]
print("답변:", response.choices[0].message.content)
print(f"첫 토큰 확신도: {math.exp(first.logprob):.0%}")
print("경쟁 후보:", [t.token for t in first.top_logprobs[1:]])

답변: 서울
첫 토큰 확신도: 100%
경쟁 후보: ['세', 'Se']


확실히 아는 질문은 **100%** 가 나오고 경쟁 후보는 사실상 0%입니다.

| 질문 | 확신도 | 뜻 |
|---|---|---|
| 대한민국의 수도는? | 100% | 확실히 안다 |
| 이 취약점의 CVE ID는? | 76% | **모르면서 찍고 있다** |

**확신도는 "이 모델이 정말 아는가"를 보여 주는 신호입니다.**

### temperature를 0으로 해도 완전히 고정되지는 않습니다

위 실험에서 `temperature=0`으로도 답이 갈릴 수 있습니다. 두 가지 이유입니다.

1. 후보 간 확률 차이가 작으면(76% vs 12%), 서버 쪽의 미세한 계산 차이만으로 순위가 뒤집힙니다.
2. OpenAI는 요청마다 **완전히 동일한 계산**을 보장하지 않습니다.

> **`temperature=0`은 일관성을 높일 뿐 정확성을 주지 않습니다.**
> 틀린 답을 **일관되게** 내놓을 수도 있습니다.

### 실무에서 쓰는 법

- 중요한 사실을 묻는 곳에서는 **확신도가 낮으면 답을 쓰지 않고** 검색이나 API 조회로 넘깁니다.
- 확신도가 낮은 단계의 결과를 **다음 단계로 넘기지 않는 것**만으로도 오류 전파를 크게 줄일 수 있습니다.

## 정리

### 이 노트북에서 배운 것

| 개념 | 설명 |
|---|---|
| 역할 분리 | 한 모델에 다 시키지 않고, 단계마다 역할을 정해 맡긴다 |
| 체인 | 앞 단계의 출력을 다음 단계의 입력으로 넘긴다 (f-string이면 충분) |
| 출력 형식 지정 | 중간 단계는 "ID만" 처럼 **좁게** 답하도록 지시한다 |
| 모델 교체 | `base_url`만 바꾸면 세 회사 모델을 같은 코드로 부를 수 있다 |
| 확신도 확인 | `logprobs=True`로 모델이 **아는지 찍는지** 구분할 수 있다 (Step 6) |

### ⚠️ 그런데 이 결과를 그대로 쓰면 안 됩니다

저장된 실행 결과를 다시 보세요. **답이 서로 어긋나 있습니다.**

| 확인할 곳 | 내용 |
|---|---|
| 1단계 첫 실행 | `CVE-2023-2031` |
| 1단계 다른 실행 | `CVE-2023-2146` ← **같은 질문인데 답이 다름** |
| 3단계 최종 답변 | "CVE-2023-2146: **OpenSSL**의 Type Confusion 취약점" |

원래 질문은 **Google Chromium V8** 취약점에 대한 것이었는데, 최종 답변은 **OpenSSL** 이야기를 하고 있습니다.
영향 버전 범위(`3.0.0 ~ 3.0.7`)와 점수(`7.5`)까지 구체적으로 적혀 있지만, **근거가 없습니다.**

**왜 이런 일이 생길까요?**

1. **이 체인에는 검색이 없습니다.** 세 모델 모두 "기억"에만 의존해 답합니다.
   CVE 번호처럼 외워야 하는 사실은 LLM이 가장 자주 틀리는 종류입니다.
2. **오류가 그대로 전달됩니다.** 1단계가 틀린 ID를 내놓으면, 2·3단계는 그것을 **사실로 받아들이고** 이야기를 이어 갑니다.
   체인이 길어질수록 처음의 작은 오류가 커집니다.
3. 게다가 3단계는 "틀렸다"고 말하지 않습니다. **그럴듯하게** 답합니다. 이것이 가장 위험한 부분입니다.

**어떻게 고치나**

| 방법 | 설명 | 실습 파일 |
|---|---|---|
| 실제 데이터를 검색해서 넣기 | NVD·MITRE의 진짜 CVE·CWE 문서를 찾아 근거로 준다 | `../rag/security_rag/` |
| 도구를 붙이기 | LLM이 직접 API를 조회하게 한다 | `simple_tool.ipynb` |
| 중간 결과 검증 | ID 형식·존재 여부를 코드로 확인한 뒤 다음 단계로 넘긴다 | — |

> **체인의 중간 결과는 반드시 확인하세요.**
> 이 노트북은 "여러 LLM을 잇는 방법"을 보여 주는 동시에, **왜 RAG가 필요한지**도 보여 줍니다.

## 스스로 해 보기

1. Step 3의 셀을 세 번 연속 실행해 보세요. 같은 CVE ID가 나오나요?
2. `temperature=0`을 세 호출 모두에 추가하면 결과가 일정해지나요? 일정해지면 **정확해진** 걸까요?
3. 1단계에서 받은 CVE ID가 `CVE-숫자-숫자` 형식인지 검사하는 코드를 넣어 보세요.
4. 세 단계의 모델을 서로 바꿔 보세요(예: 1단계를 Claude로). 답이 달라지나요?
5. `../rag/security_rag/`의 실습과 비교해 보세요. 그쪽은 실제 MITRE 문서를 근거로 답합니다.